[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/03_pretrained_korean/03_pretrained_korean_solutions.ipynb)

# 03. 사전 학습 한국어 모델 — 연습 문제 해설

[03_pretrained_korean.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/03_pretrained_korean/03_pretrained_korean.ipynb) 끝의
연습 문제 6개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **이 노트북의 숫자는 방향을 보는 용도입니다.** 분류 머리가 새로 초기화되기 때문에
> seed를 고정해도 환경(CPU/GPU, 라이브러리 버전)에 따라 소수점 이하가 달라집니다.
> 본문에서 잰 기준값은 **검증 0.8705 · 평가 0.8345** 였습니다.
>
> **문제 1은 모델이 두 배 커서 오래 걸립니다.** 본문의 `small` 모델이 CPU에서 52분
> 걸렸으니, `base`는 그보다 한참 깁니다. GPU를 켜고 시작하세요.
> 시간이 없다면 **문제 4부터** 보세요 — 학습 없이 본문 모델만 있으면 됩니다.

In [ ]:
import sys, time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q transformers pandas scikit-learn matplotlib seaborn koreanize-matplotlib

YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RANDOM_STATE = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")
data = raw[["title", "label"]].sample(20_000, random_state=RANDOM_STATE).reset_index(drop=True)
X, y_text = data["title"].values, data["label"].values
X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_valid = label_encoder.transform(y_valid_text)
N_CLASSES = len(label_encoder.classes_)

print("device:", device, "· 학습", len(X_train), "· 검증", len(X_valid))

본문의 학습 과정을 함수 하나로 묶어둡니다. 문제마다 인자만 바꿔 부릅니다.
**본문과 같은 코드**이고, 바꿀 수 있게 인자로 뺐을 뿐입니다.

In [ ]:
def finetune(model_name="klue/roberta-small", max_len=32, lr=3e-5, epochs=3,
             batch=32, n_train=None, verbose=True):
    """(모델, 토크나이저, 검증 정확도, 학습 시간)을 돌려준다."""
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=N_CLASSES).to(device)

    def encode(texts):
        e = tok(list(texts), padding="max_length", truncation=True,
                max_length=max_len, return_tensors="pt")
        return e["input_ids"], e["attention_mask"]

    xs = X_train if n_train is None else X_train[:n_train]
    ys = y_train if n_train is None else y_train[:n_train]
    ids, mask = encode(xs)
    labels = torch.tensor(ys)

    @torch.no_grad()
    def predict(texts, bs=128):
        model.eval()
        i_, m_ = encode(texts)
        out = []
        for i in range(0, len(i_), bs):
            out.append(model(input_ids=i_[i:i + bs].to(device),
                             attention_mask=m_[i:i + bs].to(device)).logits.cpu())
        return torch.cat(out)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    started = time.time()
    for ep in range(epochs):
        model.train()
        order = torch.randperm(len(ids))
        for i in range(0, len(order), batch):
            b = order[i:i + batch]
            opt.zero_grad()
            model(input_ids=ids[b].to(device), attention_mask=mask[b].to(device),
                  labels=labels[b].to(device)).loss.backward()
            opt.step()
        if verbose:
            acc = (predict(X_valid).argmax(1).numpy() == y_valid).mean()
            print(f"  epoch {ep + 1}  valid_acc={acc:.4f}  ({time.time() - started:.0f}초)")

    model.predict_logits = predict          # 문제 4·5에서 다시 씁니다
    return model, tok, (predict(X_valid).argmax(1).numpy() == y_valid).mean(), time.time() - started

---

## 문제 1. `klue/roberta-base`로 바꾸면?

`MODEL_NAME` 한 줄만 바꿉니다. 정확도와 **학습 시간**을 함께 재는 것이 이 문제의 핵심입니다.

In [ ]:
from transformers import AutoConfig

for name in ["klue/roberta-small", "klue/roberta-base"]:
    cfg = AutoConfig.from_pretrained(name)
    m, _, acc, secs = finetune(model_name=name, verbose=False)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:<22} 정확도 {acc:.4f}  층 {cfg.num_hidden_layers:>2}개"
          f"  파라미터 {n_params/1e6:>5.1f}M  {secs:>5.0f}초")
    del m

**결과는 직접 재서 보세요. 이 해설은 답을 미리 말하지 않습니다.**

말할 수 있는 것은 구조 차이뿐입니다. `base`는 층이 12개, `small`은 6개이고
파라미터도 그만큼 차이가 납니다. 층이 많으면 한국어를 더 촘촘히 담을 여지가 있고,
KLUE-YNAT는 원래 이런 모델들을 재려고 만든 데이터셋이라 논문에 두 모델의 점수가 실려 있습니다.
**그래도 여러분의 분할·여러분의 seed에서 어떻게 나올지는 재봐야 압니다.**

**이 문제의 핵심은 "얼마나 더 높은가"가 아니라 "그만큼의 값어치가 있는가"입니다.**
파라미터가 늘어난 만큼 학습·추론 시간도 늘어납니다. 본문 9절의 표를 다시 보세요.
정확도 한 칸만 보고 고르는 것이 아닙니다.

**판단 기준** — 01번 7절에서 배운 대로 **분할을 바꾸면 정확도가 0.01씩 흔들립니다.**
두 모델의 차이가 그 폭 안에 있다면, seed를 바꿔 서너 번 돌려 **차이의 방향이 일관되는지**
확인한 뒤에 판단하세요. 한 번 재서 나온 차이로 두 배 큰 모델을 고르는 것은 위험합니다.

> **본문 8절이 바로 이 함정의 사례였습니다.** 낙폭이 줄어든 것을 보고 "강건하다"고
> 결론 내릴 뻔했는데, 주제별로 쪼개보니 다른 이야기였습니다. 요약된 숫자 하나로
> 모델을 고르지 마세요.

---

## 문제 2. `MAX_LEN`을 16으로 줄이면?

**자르기 전에 먼저 확인합니다.** 02번 연습 문제 1번의 교훈이 그것이었습니다 —
길이를 줄이면 **뒤가 잘립니다.** 몇 %가 잘리는지 보고 결정하세요.

In [ ]:
tok = AutoTokenizer.from_pretrained("klue/roberta-small")
길이 = pd.Series([len(tok(s)["input_ids"]) for s in X_train])

print(길이.describe().round(1))
for q in [0.90, 0.95, 0.99]:
    print(f"  {q:.0%} 지점: {길이.quantile(q):.0f} 토큰")
print(f"\n16토큰을 넘는 제목: {(길이 > 16).mean():.1%}")
print(f"32토큰을 넘는 제목: {(길이 > 32).mean():.1%}")

In [ ]:
for max_len in [16, 32]:
    _, _, acc, secs = finetune(max_len=max_len, verbose=False)
    print(f"MAX_LEN={max_len:>2}  정확도 {acc:.4f}  {secs:.0f}초")

**분위수를 먼저 본 것이 핵심입니다.** CPU에서 재보면 이렇게 나옵니다.

| | 값 |
|---|---|
| 평균 길이 | 15.4 토큰 |
| 95% 지점 | 20 토큰 |
| 99% 지점 | 23 토큰 |
| **가장 긴 제목** | **29 토큰** |
| 16토큰을 넘는 비율 | **36.5%** |
| 32토큰을 넘는 비율 | **0.00%** |

**`MAX_LEN=32`는 단 한 건도 자르지 않습니다.** 반면 16으로 줄이면 **열에 서너 건이 잘립니다.**
잘린 만큼 정확도가 내려가고, 대신 학습 시간은 줄어듭니다. 어느 쪽을 택할지는
"얼마나 잘리는가"를 보고 정하는 것이지, 감으로 정하는 것이 아닙니다.

**서브워드는 단어보다 토큰이 많다**는 점에 주의하세요. 본문 3절에서 제목 하나가
단어 단위보다 서브워드 쪽이 더 많은 토큰이 된다고 했습니다. 02번에서 `SEQ_LEN=12`가
충분했다고 해서 여기서도 12로 두면 안 되는 이유입니다. **토크나이저가 바뀌면 길이도 다시 재야 합니다.**

---

## 문제 3. 학습률을 3e-4로 열 배 키우면?

In [ ]:
# 1 epoch만 돌려도 차이가 뚜렷합니다. 예측이 몇 개 주제에 몰리는지도 함께 봅니다.
for lr in [3e-5, 3e-4]:
    m, _, acc, secs = finetune(lr=lr, epochs=1, verbose=False)
    pred = m.predict_logits(X_valid).argmax(1).numpy()
    쓰인주제 = sorted(set(pred))
    print(f"lr={lr}  검증 정확도 {acc:.4f}  "
          f"예측에 쓰인 주제 {len(쓰인주제)}/{N_CLASSES}개 "
          f"{[label_encoder.classes_[i] for i in 쓰인주제]}")
    del m

**무너집니다. 다만 "아무것도 못 배운다"와는 다른 모습입니다.** CPU에서 1 epoch씩 재본 값입니다.

| 학습률 | train_loss | 검증 정확도 | 예측에 쓰인 주제 |
|---|---|---|---|
| 3e-5 (본문) | 0.5245 | **0.8750** | 7 / 7 |
| **3e-4** | 0.9109 | **0.3765** | **4 / 7** |

정확도 0.3765는 무작위(1/7 ≈ 0.14)보다는 높습니다. **완전히 망가진 게 아니라
일부 주제를 통째로 포기한 것**입니다. 7개 중 4개만 답으로 쓰고 나머지 세 주제는
아예 예측하지 않습니다. loss도 0.91에서 더 내려가지 않습니다 —
**골짜기 바닥을 지나쳐 버려서 큰 걸음으로 왔다 갔다 하는 상태**입니다.

`ml-curriculum` 01번에서 배운 "학습률이 너무 크면 목표를 지나쳐 버린다"와 같은 현상입니다.
**그런데 파인튜닝에서는 잃는 것이 다릅니다.** 처음부터 배우는 모델이라면 다시 배우면 그만인데,
여기서는 **사전 학습으로 얻은 한국어 지식을 크게 흔들어 망가뜨립니다.**
이것을 [치명적 망각](https://github.com/karzit/temp/blob/master/glossary.md#catastrophic-forgetting)(catastrophic forgetting)이라고 부릅니다.
남이 며칠 걸려 만들어둔 것을 한 epoch 만에 지우는 셈입니다.

파인튜닝의 학습률이 2e-5~5e-5처럼 작은 이유가 이것입니다.
**우리는 새로 배우는 것이 아니라 이미 아는 것을 조금 조정하는 중입니다.**

> **진단할 때 정확도만 보지 마세요.** 0.3765만 보면 "덜 배웠나 보다" 싶어 epoch를
> 늘리게 됩니다. **예측에 쓰인 주제가 4개뿐**이라는 것을 봐야 학습률 문제인 줄 압니다.
> 본문 8절에서 요약된 숫자 하나를 쪼개봤던 것과 같은 이야기입니다.

---

## 문제 4. 이 모델도 같은 곳에서 틀리는가

**학습이 필요 없는 문제입니다.** 본문에서 학습한 모델을 그대로 씁니다.
01번 10~11절에서 TF-IDF가 헷갈리던 주제 쌍을 이 모델도 헷갈리는지 봅니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    pass

# 사전 학습 모델 (없으면 여기서 한 번 학습합니다)
model, tok, acc, _ = finetune(verbose=False)
pred_nn = model.predict_logits(X_valid).argmax(1).numpy()

# 01번의 TF-IDF 모델을 같은 자리에 다시 세웁니다
tfidf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train, y_train_text)
pred_tfidf = label_encoder.transform(tfidf.predict(X_valid))

labels = list(label_encoder.classes_)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, pred, title in [(axes[0], pred_tfidf, f"01번 TF-IDF ({(pred_tfidf == y_valid).mean():.4f})"),
                        (axes[1], pred_nn, f"03번 사전 학습 모델 ({(pred_nn == y_valid).mean():.4f})")]:
    sns.heatmap(confusion_matrix(y_valid, pred), annot=True, fmt="d", cmap="Blues",
                cbar=False, xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("예측")
    ax.set_ylabel("실제")
plt.tight_layout()
plt.show()

In [ ]:
# 둘 다 틀린 것과, 한쪽만 틀린 것을 세어봅니다.
둘다맞음 = ((pred_tfidf == y_valid) & (pred_nn == y_valid)).sum()
둘다틀림 = ((pred_tfidf != y_valid) & (pred_nn != y_valid)).sum()
tfidf만틀림 = ((pred_tfidf != y_valid) & (pred_nn == y_valid)).sum()
nn만틀림 = ((pred_tfidf == y_valid) & (pred_nn != y_valid)).sum()

print(f"둘 다 맞음      {둘다맞음:>5}")
print(f"둘 다 틀림      {둘다틀림:>5}   ← 모델을 바꿔도 구제되지 않는 행")
print(f"TF-IDF만 틀림   {tfidf만틀림:>5}")
print(f"사전학습만 틀림  {nn만틀림:>5}")

**"둘 다 틀림"이 이 문제의 답입니다.**

두 모델은 완전히 다른 방식으로 만들어졌는데도 **상당수를 같이 틀립니다.**
방식이 다른 두 모델이 같은 행에서 넘어진다면, 그것은 **모델의 한계가 아니라 데이터의 성질**입니다.

01번 3절을 떠올려보세요. **주석자 세 명이 36%에서 2:1로 갈렸습니다.** 사람도 답이 갈리는 문장을
모델이 맞히기를 기대할 수 없습니다. 혼동 행렬에서 `사회`로 오답이 흩어지는 모양도
양쪽에 똑같이 남아 있을 것입니다 — 01번에서 "딱 떨어지지 않는 것이 모이는 칸"이라고 부른 그것입니다.

**실무에서 이 계산이 중요한 이유:** "둘 다 틀림"이 전체 오답의 대부분이라면,
모델을 더 바꿔봐야 남는 이득이 얼마 없다는 뜻입니다. 그때는 모델이 아니라
**라벨이나 입력(제목 대신 본문)을 손봐야** 합니다.

---

## 문제 5. 앙상블은 이번에는 오를까?

02번 6절에서는 TF-IDF(0.8455)와 신경망(0.7927)을 섞었더니 **0.8225로 내려갔습니다.**
실력 차이가 컸기 때문입니다. 이번에는 두 모델의 실력이 더 가깝습니다.

In [ ]:
import torch.nn.functional as Fn

proba_nn = Fn.softmax(model.predict_logits(X_valid), dim=1).numpy()
proba_tfidf = tfidf.predict_proba(X_valid)   # 클래스 순서가 LabelEncoder와 같습니다

for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
    mixed = (w * proba_nn + (1 - w) * proba_tfidf).argmax(axis=1)
    tag = {0.0: "  (TF-IDF만)", 1.0: "  (사전학습만)"}.get(w, "")
    print(f"사전학습 비중 {w:.1f}   정확도 {(mixed == y_valid).mean():.4f}{tag}")

**02번의 교훈 두 가지를 여기서 다시 확인합니다.**

앙상블이 이득을 보려면 ① 서로 다른 실수를 하고 ② **실력이 엇비슷해야** 합니다.
02번에서는 ②가 깨져서 평균이 약한 쪽에 발목을 잡혔습니다.

이번에는 실력이 가까우니 조건 ②는 나아졌습니다. 그런데 **문제 4에서 센 "둘 다 틀림"을
떠올려보세요.** 두 모델이 같은 행에서 넘어진다면 조건 ①이 약한 것이고,
그 행들은 섞어도 구제되지 않습니다.

**섞어서 올랐다면** 두 모델이 서로 다른 곳에서 틀렸다는 뜻이고,
**안 올랐다면** 문제 4에서 본 "둘 다 틀림"이 그 이유입니다. 어느 쪽이 나왔든
**숫자 하나만 보고 끝내지 말고 문제 4의 결과와 이어서 읽으세요.**

그리고 올랐더라도 01번 7절의 기준을 적용하세요 — **분할 하나에서 잰 +0.002는 아직 아무 말도
못 합니다.** 두 모델을 다 유지하는 비용(메모리, 추론 시간 두 배)을 치를 만한 차이인지도 따져야 합니다.

---

## 문제 6. 자신의 데이터로

바꿀 것은 데이터를 만드는 부분뿐입니다. 나머지 코드는 그대로입니다.

In [ ]:
# 예시 — CSV 한 장에서 시작하는 경우
#
# my = pd.read_csv("my_data.csv")            # 컬럼: text, label
# X = my["text"].values
# y_text = my["label"].values
#
# X_train, X_valid, y_train_text, y_valid_text = train_test_split(
#     X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE)
# label_encoder = LabelEncoder()
# y_train = label_encoder.fit_transform(y_train_text)
# y_valid = label_encoder.transform(y_valid_text)
# N_CLASSES = len(label_encoder.classes_)
#
# finetune()      # 이 아래로는 바꿀 것이 없습니다

print("먼저 확인할 것 — 01번 2~4절에서 한 점검을 그대로 반복하세요")
print("  1. 라벨 분포 (한 범주가 몇 %인가)")
print("  2. 같은 입력에 다른 라벨이 붙은 행이 있는가")
print("  3. 최빈 라벨로 전부 찍었을 때의 정확도  ← 이것이 기준선입니다")
print("  4. 길이 분포 (MAX_LEN을 얼마로 둘지)")

**순서를 지키세요. 사전 학습 모델부터 꺼내면 안 됩니다.**

이 시리즈의 순서가 곧 답입니다.

1. **기준선부터** — 최빈 라벨로 찍었을 때 정확도가 0.9라면, 0.91짜리 모델은 아무것도 한 게 없습니다
2. **TF-IDF + 로지스틱 회귀** — 몇 초면 끝납니다. 이걸 넘지 못하는 딥러닝은 쓸 이유가 없습니다
3. **그다음에 이 노트북**

**라벨이 적다면 순서가 바뀔 수 있습니다.** 본문 9절에서 학습 건수를 줄여봤습니다.
라벨을 몇 백~몇 천 건밖에 못 만드는 상황이라면, TF-IDF보다 사전 학습 모델 쪽이
먼저 쓸 만해지는 지점이 옵니다. 그래도 **기준선(1번)은 건너뛰지 마세요.**

마지막으로 01번 3절의 이야기를 다시 짚습니다. **자신의 데이터에도 라벨 모호성이 있습니다.**
같은 문의를 두 사람이 다르게 분류하는 일은 흔합니다. 그 비율을 모르면
**목표 정확도를 어디에 둘지 정할 수 없습니다.** 몇 백 건을 두 사람이 각각 분류해보고
얼마나 갈리는지 재보는 것이, 모델을 바꾸는 것보다 먼저 할 일일 때가 많습니다.